In [3]:
!nvidia-smi

Fri Sep  4 14:13:09 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
!pip install -q fastapi uvicorn pydantic aiohttp pyngrok huggingface_hub
!CMAKE_ARGS="-DGGML_CUDA=on" pip install -q llama-cpp-python --no-cache-dir

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.9/74.9 MB 238.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 271.6 MB/s eta 0:00:00


In [7]:
!hf download Qwen/Qwen2.5-1.5B-Instruct-GGUF \
    qwen2.5-1.5b-instruct-q4_k_m.gguf --local-dir ./models


Hint: The `hf-cli` skill is not installed. Run `hf skills add -g --claude` to teach your AI agents how to use the `hf` CLI.

qwen2.5-1.5b-instruct-q4_k_m.gguf: downloading bytes:  12% 129M/1.12G [00:01<00:05, 167MB/s, 8.17MB/s  ] 
qwen2.5-1.5b-instruct-q4_k_m.gguf: downloading bytes:  24% 274M/1.12G [00:02<00:02, 287MB/s, 21.5MB/s  ]
qwen2.5-1.5b-instruct-q4_k_m.gguf: downloading bytes:  37% 419M/1.12G [00:02<00:02, 333MB/s, 34.5MB/s  ]
qwen2.5-1.5b-instruct-q4_k_m.gguf: downloading bytes:  48% 539M/1.12G [00:03<00:01, 442MB/s, 43.4MB/s  ]
qwen2.5-1.5b-instruct-q4_k_m.gguf: downloading bytes:  53% 589M/1.12G [00:03<00:01, 451MB/s, 48.3MB/s  ]
qwen2.5-1.5b-instruct-q4_k_m.gguf: downloading bytes:  66% 736M/1.12G [00:03<00:00, 389MB/s, 61.7MB/s  ]
qwen2.5-1.5b-instruct-q4_k_m.gguf: downloading bytes:  72% 800M/1.12G [00:03<00:00, 353MB/s, 66.2MB/s  ]
qwen2.5-1.5b-instruct-q4_k_m.gguf: downloading bytes:  80% 899M/1.12G [00:04<00:00, 334MB/s, 73.3MB/s  ]
qwen2.5-1.5b-instruct-q4_k_m.gguf:

In [26]:
from google.colab import files
uploaded = files.upload()

Saving local_server.py to local_server.py


In [27]:
import os, subprocess, time

env = os.environ.copy()
env["MODEL_PATH"] = "./models/qwen2.5-1.5b-instruct-q4_k_m.gguf"
env["USE_GPU"] = "1"
env["N_THREADS"] = "4"  # less relevant on GPU, but still used for prompt handling

log_file = open("server.log", "w")
proc = subprocess.Popen(
    ["uvicorn", "local_server:app", "--host", "0.0.0.0", "--port", "8000"],
    env=env, stdout=log_file, stderr=subprocess.STDOUT,
)
time.sleep(8)  # give the model time to load
print(open("server.log").read())

INFO:     Started server process [29923]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)



In [28]:
import requests
r = requests.post("http://localhost:8000/generate",
                   json={"prompt": "Say hello in one sentence.", "max_tokens": 20})
print(r.json())

{'text': 'Hello! How can I assist you today?', 'ttft': 0.20602384900030302, 'total_latency': 0.3278144659998361, 'tokens_genrated': 9, 'prompt_tokens': 6}


In [29]:
from pyngrok import ngrok

NGROK_TOKEN = os.environ.get("NGROK_AUTHTOKEN", "YOUR_NGROK_AUTHTOKEN")
ngrok.set_auth_token(NGROK_TOKEN)
public_url = ngrok.connect(8000, "http")
print("Public URL:", public_url)

Public URL: NgrokTunnel: "https://cocoa-chuck-chrome.ngrok-free.dev" -> "http://localhost:8000"
